# Learned vs Fixed Metric — Small Examples Comparison

This notebook compares the **learnable inverse-metric** optimizers you added to the repo
against the **original fixed-metric** versions, using the same low-D benchmark functions
(Ackley, Beale, Himmelblau, Rastrigin, Rosenbrock) that back the figures and the original
`small_examples.ipynb` in the paper. It mirrors the repo’s structure and naming:

- Fixed (JAX/Optax): `optimisers/jax_fixed.py` exporting
  `custom_sgd`, `custom_sgd_log`, `custom_sgd_rms`  
- Learnable (JAX/Optax): `optimisers/jax_learnable.py` exporting
  `custom_sgd_learnable_diag`, `custom_sgd_log_learnable_diag`
- Optional Torch mirrors: `optimisers/torch_fixed.py`, `optimisers/torch_learnable.py`

> Assumption: **you provide tuned hyperparameters** for each optimizer (per your note).
> The cells below have clearly marked dicts for your best settings.

**Reference (algorithms & EMA’d denominator):** Appendix A / Listing 1 in the paper
describes the fixed variants and their Optax interfaces; we use the same API surface here.

In [ ]:
# Imports (defer heavy deps until actually running experiments)
import importlib, sys, time, math
import numpy as np

def _try(name):
    try:
        mod = importlib.import_module(name)
        print(f"OK: {name}")
        return mod
    except Exception as e:
        print(f"Missing: {name} -> {e}")
        return None

# Core (JAX-side; small-examples in the paper used JAX/Optax)
jax = _try("jax")
jnp = _try("jax.numpy")
optax = _try("optax")

# Our optimisers
jax_fixed = _try("optimisers.jax_fixed")            # custom_sgd, custom_sgd_log, custom_sgd_rms
jax_learn = _try("optimisers.jax_learnable")        # custom_sgd_learnable_diag, custom_sgd_log_learnable_diag

# Optional: PyTorch ports
torch = _try("torch")
torch_fixed = _try("optimisers.torch_fixed")
torch_learn = _try("optimisers.torch_learnable")

print("Ready.")

## Benchmark functions

We reproduce the five standard low-D functions used in the paper’s *small examples* section,
with known global minima for simple early-stopping:

- **Rosenbrock**: min at (1, 1), f = 0  
- **Rastrigin**: min at (0, 0), f = 0  
- **Himmelblau**: minima include (3, 2), f = 0  
- **Beale**: min at (3, 0.5), f = 0  
- **Ackley**: min at (0, 0), f = 0

In [ ]:
# JAX versions (vectorized over theta = [x, y])
if jnp:
    def rosenbrock(theta, a=1.0, b=100.0):
        x, y = theta[0], theta[1]
        return (a - x)**2 + b*(y - x**2)**2

    def rastrigin(theta, A=10.0):
        x, y = theta[0], theta[1]
        return 2*A + (x**2 - A*jnp.cos(2*jnp.pi*x)) + (y**2 - A*jnp.cos(2*jnp.pi*y))

    def himmelblau(theta):
        x, y = theta[0], theta[1]
        return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

    def beale(theta):
        x, y = theta[0], theta[1]
        term1 = (1.5 - x + x*y)
        term2 = (2.25 - x + x*y**2)
        term3 = (2.625 - x + x*y**3)
        return term1**2 + term2**2 + term3**2

    def ackley(theta, a=20.0, b=0.2, c=2*jnp.pi):
        x, y = theta[0], theta[1]
        term1 = -a * jnp.exp(-b * jnp.sqrt((x**2 + y**2)/2.0))
        term2 = -jnp.exp((jnp.cos(c*x) + jnp.cos(c*y))/2.0)
        return term1 + term2 + a + jnp.e

    BENCHES = {
        "rosenbrock": {"fn": rosenbrock,  "theta_star": jnp.array([1.0, 0.99999999]), "f_star": 0.0},
        "rastrigin":  {"fn": rastrigin,   "theta_star": jnp.array([0.0, 0.0]),         "f_star": 0.0},
        "himmelblau": {"fn": himmelblau,  "theta_star": jnp.array([3.0, 2.0]),         "f_star": 0.0},
        "beale":      {"fn": beale,       "theta_star": jnp.array([3.0, 0.5]),          "f_star": 0.0},
        "ackley":     {"fn": ackley,      "theta_star": jnp.array([0.0, 0.0]),          "f_star": 0.0},
    }
else:
    BENCHES = {}

## Hyperparameters (fill with your tuned values)

As per your note, we **assume you will provide optimal hyperparameters** for each optimizer.
Below are placeholders with reasonable defaults; **please overwrite** for your runs.

In [ ]:
# Fixed-metric (JAX) — per paper Algorithms / Listing 1
HP_FIXED = dict(
    # Plain (Alg. 1)
    custom_sgd      = dict(learning_rate=0.05, momentum=0.9, xi=0.1, beta=0.8, weight_decay=0.0),
    # Log-loss (Alg. 2) — only used if you enable it below
    custom_sgd_log  = dict(learning_rate=0.05, momentum=0.9, xi=0.1, beta=0.8, weight_decay=0.0),
    # RMS variant (optional)
    custom_sgd_rms  = dict(learning_rate=0.05, momentum=0.9, xi=0.1, beta=0.8, beta_rms=0.99, weight_decay=0.0, eps=1e-8),
)

# Learnable-metric (JAX) — diagonal inverse metric
HP_LEARN = dict(
    # Plain + learnable diag gamma^{-1}
    custom_sgd_learnable_diag     = dict(learning_rate=0.05, momentum=0.9, xi=0.1, beta=0.8,
                                         weight_decay=0.0, metric_lr=5e-4, metric_reg=1e-4, metric_clip=4.0),
    # Log-loss + learnable diag gamma^{-1} (requires loss)
    custom_sgd_log_learnable_diag = dict(learning_rate=0.05, momentum=0.9, xi=0.1, beta=0.8,
                                         weight_decay=0.0, metric_lr=5e-4, metric_reg=1e-4, metric_clip=4.0),
)

# Which variants to compare on the small examples
USE_LOG_LOSS = False        # set True to compare log-loss versions as well
USE_RMS      = False        # optional: fixed RMS variant vs learnable (not provided)

## JAX runner

We keep the training loop close to the repo’s JAX/Optax style (Alg. 1/2): compute gradients,
build the preconditioned step via the optimizer’s `update`, apply, and (for log-loss variants)
pass the scalar loss into `update`.

We measure both **iterations** and **wall-time** to replicate the paper’s plots.

In [ ]:
if jnp and jax_fixed and jax_learn:
    import jax
    import jax.numpy as jnp
    import optax

    def _jit_value_and_grad(f):
        return jax.jit(jax.value_and_grad(f))

    def run_one_function_jax(fn_name, theta0, steps=2000, tol=1e-10, seed=0):
        """
        Compare fixed vs learnable on a single function.
        Returns dict with loss curves and runtime for both variants.
        """
        bench = BENCHES[fn_name]
        f = bench["fn"]
        f_star = float(bench["f_star"])

        # Loss & grad
        vgf = _jit_value_and_grad(lambda th: f(th))

        # Build optims
        fixed_builder  = jax_fixed.custom_sgd(**HP_FIXED["custom_sgd"])
        learn_builder  = jax_learn.custom_sgd_learnable_diag(**HP_LEARN["custom_sgd_learnable_diag"])

        # Initial state
        th_f = jnp.array(theta0)
        st_f = fixed_builder.init(th_f)

        th_l = jnp.array(theta0)
        st_l = learn_builder.init(th_l)

        # Logs
        out = dict(fixed_loss=[], learned_loss=[], fixed_ms=None, learned_ms=None,
                   fixed_steps=None, learned_steps=None)

        # ---- Fixed
        t0 = time.time()
        for t in range(1, steps+1):
            loss, grad = vgf(th_f)
            updates, st_f = fixed_builder.update(grad, st_f, params=th_f)
            th_f = optax.apply_updates(th_f, updates)
            out["fixed_loss"].append(float(loss))
            if abs(float(loss - f_star)) <= tol:
                out["fixed_steps"] = t
                break
        t1 = time.time()
        out["fixed_ms"] = 1000.0*(t1 - t0)
        if out["fixed_steps"] is None:
            out["fixed_steps"] = steps

        # ---- Learnable
        t0 = time.time()
        for t in range(1, steps+1):
            loss, grad = vgf(th_l)
            updates, st_l = learn_builder.update(grad, st_l, params=th_l)
            th_l = optax.apply_updates(th_l, updates)
            out["learned_loss"].append(float(loss))
            if abs(float(loss - f_star)) <= tol:
                out["learned_steps"] = t
                break
        t1 = time.time()
        out["learned_ms"] = 1000.0*(t1 - t0)
        if out["learned_steps"] is None:
            out["learned_steps"] = steps

        return out

else:
    print("Skipping JAX runner — missing jax/optax or optimisers.")

## Run the comparison

Set starting points reasonably far from the minima (same seed across methods).
We record **loss vs iteration**, **iterations to tolerance**, and **wall-time**.

In [ ]:
RESULTS = {}

if jnp and jax_fixed and jax_learn:
    # A few nontrivial starts; feel free to add more
    starts = {
        "rosenbrock": np.array([-1.2, 1.0], dtype=np.float32),
        "rastrigin":  np.array([ 3.0,  4.0], dtype=np.float32),
        "himmelblau": np.array([-2.0, 2.0], dtype=np.float32),
        "beale":      np.array([-1.0, 1.0], dtype=np.float32),
        "ackley":     np.array([ 2.0,  2.0], dtype=np.float32),
    }
    for name, s0 in starts.items():
        print("Running:", name)
        RESULTS[name] = run_one_function_jax(name, s0, steps=2000, tol=1e-10)
else:
    print("JAX comparisons not available.")

## Plots

For each function we show:
- **Loss vs iteration** (log-scale)
- **Iterations & wall-time** summary

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(name, res):
    plt.figure()
    plt.yscale("log")
    plt.plot(res["fixed_loss"], label="fixed")
    plt.plot(res["learned_loss"], label="learned")
    plt.title(f"{name}: loss vs iteration")
    plt.xlabel("iteration")
    plt.ylabel("loss")
    plt.legend()
    plt.show()

def print_summary_table(results):
    import pandas as pd
    rows = []
    for k, v in results.items():
        rows.append(dict(
            function = k,
            fixed_iters = v["fixed_steps"],
            learned_iters = v["learned_steps"],
            fixed_ms = round(v["fixed_ms"], 2),
            learned_ms = round(v["learned_ms"], 2),
        ))
    df = pd.DataFrame(rows).set_index("function")
    try:
        from caas_jupyter_tools import display_dataframe_to_user
        display_dataframe_to_user("learned_vs_fixed_summary", df)
    except Exception:
        print(df)
    return df

if len(RESULTS):
    for name, res in RESULTS.items():
        plot_loss_curves(name, res)
    summary_df = print_summary_table(RESULTS)
else:
    print("No results to plot.")

## (Optional) Log-loss variants

Enable `USE_LOG_LOSS = True` earlier to also compare `custom_sgd_log` vs
`custom_sgd_log_learnable_diag`. The runner below matches Algorithm 2’s API
by passing the current scalar loss into `update`.

In [ ]:
if USE_LOG_LOSS and jnp and jax_fixed and jax_learn:
    import jax
    import jax.numpy as jnp
    import optax

    def run_one_function_jax_log(fn_name, theta0, steps=2000, tol=1e-10, seed=0):
        bench = BENCHES[fn_name]
        f = bench["fn"]
        f_star = float(bench["f_star"])

        vgf = jax.jit(jax.value_and_grad(lambda th: f(th)))

        fixed_builder  = jax_fixed.custom_sgd_log(**HP_FIXED["custom_sgd_log"])
        learn_builder  = jax_learn.custom_sgd_log_learnable_diag(**HP_LEARN["custom_sgd_log_learnable_diag"])

        th_f = jnp.array(theta0); st_f = fixed_builder.init(th_f)
        th_l = jnp.array(theta0); st_l = learn_builder.init(th_l)

        out = dict(fixed_loss=[], learned_loss=[], fixed_ms=None, learned_ms=None,
                   fixed_steps=None, learned_steps=None)

        t0=time.time()
        for t in range(1, steps+1):
            loss, grad = vgf(th_f)
            updates, st_f = fixed_builder.update(grad, st_f, loss, params=th_f)
            th_f = optax.apply_updates(th_f, updates)
            out["fixed_loss"].append(float(loss))
            if abs(float(loss - f_star)) <= tol:
                out["fixed_steps"] = t; break
        out["fixed_ms"] = 1000*(time.time()-t0)
        if out["fixed_steps"] is None: out["fixed_steps"] = steps

        t0=time.time()
        for t in range(1, steps+1):
            loss, grad = vgf(th_l)
            updates, st_l = learn_builder.update(grad, st_l, loss, params=th_l)
            th_l = optax.apply_updates(th_l, updates)
            out["learned_loss"].append(float(loss))
            if abs(float(loss - f_star)) <= tol:
                out["learned_steps"] = t; break
        out["learned_ms"] = 1000*(time.time()-t0)
        if out["learned_steps"] is None: out["learned_steps"] = steps

        return out

    RESULTS_LOG = {}
    starts = {
        "rosenbrock": np.array([-1.2, 1.0], dtype=np.float32),
        "rastrigin":  np.array([ 3.0,  4.0], dtype=np.float32),
        "himmelblau": np.array([-2.0, 2.0], dtype=np.float32),
        "beale":      np.array([-1.0, 1.0], dtype=np.float32),
        "ackley":     np.array([ 2.0,  2.0], dtype=np.float32),
    }
    for name, s0 in starts.items():
        print("Running (log-loss):", name)
        RESULTS_LOG[name] = run_one_function_jax_log(name, s0, steps=2000, tol=1e-10)

    for name, res in RESULTS_LOG.items():
        plot_loss_curves(name + " (log)", res)
    _ = print_summary_table(RESULTS_LOG)
else:
    if USE_LOG_LOSS:
        print("Log-loss requested but JAX/optimisers missing.")

## (Optional) Tiny Torch regression (sanity)

A tiny linear regression on random data to confirm the Torch learnable metric also
trains and the metric moves. This is *not* part of the small examples; it’s just a quick sanity check.

In [ ]:
if torch and torch_learn:
    import torch, torch.nn as nn, torch.nn.functional as F
    from optimisers.torch_learnable import SGDLearnableDiag

    X = torch.randn(1024, 10)
    w_true = torch.randn(10, 1)
    b_true = torch.randn(1)
    y = X @ w_true + b_true + 0.05*torch.randn(1024,1)

    model = nn.Linear(10,1)
    opt = SGDLearnableDiag(model.parameters(), lr=0.05, momentum=0.9, xi=0.1, beta=0.8,
                           weight_decay=0.0, metric_lr=5e-4, metric_reg=1e-4, metric_clip=4.0, log_loss=False)

    losses, sstd = [], []
    for t in range(200):
        opt.zero_grad(set_to_none=True)
        pred = model(X)
        loss = F.mse_loss(pred, y)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
        # Track movement of s on first tensor
        for p in model.parameters():
            s = opt.state[p].get("log_diag", None)
            if s is not None:
                sstd.append(float(s.std().detach()))
                break

    import matplotlib.pyplot as plt
    plt.figure(); plt.plot(losses); plt.yscale("log")
    plt.title("Torch: loss vs step (learnable)"); plt.xlabel("step"); plt.ylabel("loss")
    plt.show()

    plt.figure(); plt.plot(sstd); plt.title("Torch: std(s) vs step");
    plt.xlabel("step"); plt.ylabel("std(s)"); plt.show()

else:
    print("Torch learnable optimiser not available — skipping.")
"

### Notes

- The **fixed** variants correspond to Algorithms 1/2 and the RMS modification in the paper’s Appendix A / Listing 1 (Optax-style API).
- The **learnable** variants keep the same update and denominator EMA but replace the fixed inverse metric with a **diagonal learnable** parameterisation, updated online with a tiny step that uses the same terms already computed by the algorithms.

Happy benchmarking!